In [1]:
import math
import random
from PIL import Image
from pathlib import Path
from tqdm.auto import tqdm

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms

from torchinfo import summary
from transformers import ConvNextV2ForImageClassification

from lib.muon import Muon
from src.utils import query_gallery_split
from src.metrics import evaluate

In [2]:
torch._dynamo.config.recompile_limit = 16

In [3]:
seed = 56
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

In [4]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

In [5]:
class CarsDataset(torch.utils.data.Dataset):
    def __init__(self, root_path, data, transform=None):
        self.root_path = root_path
        self.data = data
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):
        img_path = self.root_path / self.data.loc[index, "image_path"]
        assert img_path.is_file()
        try:
            image = torchvision.io.read_image(img_path)
        except RuntimeError:
            image = Image.open(img_path)
            image = transforms.functional.pil_to_tensor(image)

        x1, y1, x2, y2 = self.data.loc[index, ["x1", "y1", "x2", "y2"]]
        if 0 <= x1 < x2 <= image.shape[2] and 0 <= y1 < y2 <= image.shape[1]:
            image = image[:, y1:y2, x1:x2]

        if image.size(0) == 4:
            image = image[:3]
        elif image.size(0) == 1:
            image = image.repeat(3, 1, 1)

        if self.transform is not None:
            image = self.transform(image)

        return image, self.data.loc[index, "class"], self.data.loc[index, "weight"]

In [6]:
class InferenceDataset(torch.utils.data.Dataset):
    def __init__(self, root_path, data, transform=None):
        self.root_path = root_path
        self.data = data[["image_id", "x", "y", "w", "h"]].to_numpy()
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):
        row = self.data[index]

        img_path = (self.root_path / row[0]).with_suffix(".jpg")
        assert img_path.is_file()
        image = torchvision.io.read_image(img_path)

        x, y, w, h = row[1:]
        image = image[:, y : y + h, x : x + w]
        assert image.shape == (3, h, w)

        if self.transform is not None:
            image = self.transform(image)

        return image

In [7]:
data_1 = pd.read_csv("./CompCars/boxes.csv")
data_1["class"] = data_1["image_path"].str.split("/").str.slice(0, -1).str.join("/")
data_1["image_path"] = "CompCars/data/image/" + data_1["image_path"]

data_2 = pd.read_csv("./Car-1000/boxes.csv")
data_2["class"] = data_2["image_path"].str.split("/").str.slice(0, -1).str.join("/")
data_2["image_path"] = "Car-1000/all_images/" + data_2["image_path"]

data_3 = pd.read_csv("./VMMRdb/boxes.csv")
data_3["class"] = data_3["image_path"].str.split("/").str.slice(0, -1).str.join("/")
data_3["image_path"] = "VMMRdb/" + data_3["image_path"]

train_data = pd.concat([data_1, data_2, data_3], ignore_index=True)

class2id = {}
for idx, cls in enumerate(train_data["class"].unique()):
    class2id[cls] = idx
train_data["class"] = train_data["class"].map(class2id)

train_data["weight"] = (
    1 / np.sqrt(train_data.groupby("class")["class"].transform("count"))
)
train_data["weight"] = (
    train_data["weight"] / train_data["weight"].mean()
).astype(np.float32)

train_data

,image_path,x1,y1,x2,y2,class,weight
0,CompCars/data/image/1/1101/2011/07b90decb92ba6...,124,33,674,480,0,1.802874
1,CompCars/data/image/1/1101/2011/2272c7d324cf79...,52,153,849,506,0,1.802874
2,CompCars/data/image/1/1101/2011/3a62131af5fe8e...,85,107,682,469,0,1.802874
3,CompCars/data/image/1/1101/2011/5133ca181b82af...,91,23,759,551,0,1.802874
4,CompCars/data/image/1/1101/2011/6632ec1c2f7f87...,58,134,707,504,0,1.802874
...,...,...,...,...,...,...,...
562074,VMMRdb/willys_cj3b_1953/1953 Willys Jeep Cj-3b...,1,73,587,433,14613,2.704312
562075,VMMRdb/willys_cj3b_1953/1953 Willys Jeep Cj-3b...,113,30,556,442,14613,2.704312
562076,VMMRdb/willys_cj3b_1953/1953 Willys Jeep Cj-3b...,0,58,599,445,14613,2.704312
562077,VMMRdb/willys_cj3b_1954/1954 Willy's CJ3B_00E0...,27,85,442,329,14614,5.408623


In [8]:
data = pd.read_csv("./data/train.csv")
data

,image_id,x,y,w,h,vehicle_id,camera_id
0,f0d68d754a8a444b8b49628b0633068c,1199,491,619,589,403,90
1,b94be87b5ced41f5868f2dfb415100e4,0,308,437,420,403,89
2,296ab330ff8e49dd878d922c10352ca0,1199,433,682,594,403,90
3,5b7baa07e07640449243f4619a3077cc,1,317,434,414,403,89
4,ee3240d083594b5dbed8d8d28c236f91,0,332,436,395,403,89
...,...,...,...,...,...,...,...
9551,33c775ad393847f180aa715ac2d4c970,631,201,822,626,330,48
9552,56b0fd4da91545038e7228e5fe3706f0,630,201,823,624,330,48
9553,24ab438a002044ceaf3ab0cfafb811e9,1014,35,713,516,330,73
9554,f15c0a764e054d5ebf103925ce5f13ec,1016,33,707,518,330,73


In [9]:
query_df, gallery_df = query_gallery_split(data)
len(query_df), len(gallery_df)

(5560, 3996)

In [10]:
train_transforms = transforms.Compose(
    [
        transforms.Resize((224, 224), antialias=True),
        transforms.RandomHorizontalFlip(),
        transforms.TrivialAugmentWide(),
        transforms.ConvertImageDtype(torch.float),
        transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225)),
        transforms.RandomErasing(),
    ]
)
train_transforms

Compose(
    Resize(size=(224, 224), interpolation=bilinear, max_size=None, antialias=True)
    RandomHorizontalFlip(p=0.5)
    TrivialAugmentWide(num_magnitude_bins=31, interpolation=InterpolationMode.NEAREST, fill=None)
    ConvertImageDtype()
    Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225))
    RandomErasing(p=0.5, scale=(0.02, 0.33), ratio=(0.3, 3.3), value=0, inplace=False)
)

In [12]:
train_dataset = CarsDataset(Path(""), train_data, train_transforms)
train_dataset[0]

(tensor([[[-2.1179, -2.1179, -2.1179,  ..., -0.2684, -0.1314, -0.2684],
          [-2.1179, -2.1179, -2.1179,  ..., -0.2684, -0.1828, -0.3027],
          [-1.9124, -1.9638, -2.0152,  ..., -0.2513, -0.2513, -0.3541],
          ...,
          [-1.6384, -1.9124, -1.6727,  ..., -0.5424, -0.4911, -0.3027],
          [-1.4500, -1.5870, -1.5357,  ..., -0.6281, -0.6281, -0.3883],
          [-1.7240, -1.5357, -1.7754,  ..., -0.4054, -0.5424, -0.6623]],
 
         [[-2.0357, -2.0357, -2.0357,  ..., -0.3200, -0.1800, -0.3200],
          [-2.0357, -2.0357, -2.0357,  ..., -0.3200, -0.2325, -0.3375],
          [-1.9307, -1.9832, -2.0357,  ..., -0.2850, -0.2850, -0.4076],
          ...,
          [-1.5455, -1.8256, -1.5805,  ..., -1.0378, -1.0203, -0.8452],
          [-1.3529, -1.4930, -1.4405,  ..., -1.1253, -1.1604, -0.9328],
          [-1.6331, -1.4405, -1.6856,  ..., -0.8978, -1.0903, -1.2129]],
 
         [[-1.8044, -1.8044, -1.8044,  ..., -0.0964,  0.0431, -0.0964],
          [-1.8044, -1.8044,

In [13]:
val_transforms = transforms.Compose(
    [
        transforms.Resize((224, 224), antialias=True),
        transforms.ConvertImageDtype(torch.float),
        transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225)),
    ]
)
val_transforms

Compose(
    Resize(size=(224, 224), interpolation=bilinear, max_size=None, antialias=True)
    ConvertImageDtype()
    Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225))
)

In [14]:
val_dataset = torch.utils.data.ConcatDataset(
    [
        InferenceDataset(Path("data/images/"), query_df, val_transforms),
        InferenceDataset(Path("data/images/"), gallery_df, val_transforms),
    ]
)
val_dataset[0]

tensor([[[-1.2445, -1.2788, -1.3130,  ...,  0.1597,  0.8276,  0.8104],
         [-1.1075, -1.0904, -1.1589,  ..., -0.0116,  0.1426,  0.1597],
         [ 0.4508,  0.5878,  0.6906,  ...,  0.0569,  0.0398,  0.0398],
         ...,
         [ 0.1254, -0.3369, -0.5424,  ...,  0.1083,  0.0741,  0.0569],
         [ 0.1254, -0.2513, -0.5424,  ...,  0.0912,  0.0741,  0.1254],
         [ 0.1083, -0.1999, -0.5596,  ...,  0.0569,  0.0912,  0.0912]],

        [[-1.1253, -1.1604, -1.1954,  ...,  0.3452,  1.0280,  1.0105],
         [-0.9853, -0.9678, -1.0378,  ...,  0.1702,  0.3277,  0.3452],
         [ 0.6078,  0.7479,  0.8529,  ...,  0.2402,  0.2227,  0.2227],
         ...,
         [ 0.2927, -0.1275, -0.3550,  ...,  0.2577,  0.2227,  0.2052],
         [ 0.2927, -0.0399, -0.3550,  ...,  0.2402,  0.2227,  0.2752],
         [ 0.2927,  0.0126, -0.3550,  ...,  0.2052,  0.2402,  0.2402]],

        [[-0.8284, -0.8633, -0.8981,  ...,  0.6531,  1.3328,  1.3154],
         [-0.6890, -0.6715, -0.7413,  ...,  0

In [15]:
train_dataloader = torch.utils.data.DataLoader(
    dataset=train_dataset,
    batch_size=32,
    num_workers=8,
    drop_last=True,
    shuffle=True,
    pin_memory=True,
    persistent_workers=True,
)

In [16]:
val_dataloader = torch.utils.data.DataLoader(
    dataset=val_dataset,
    batch_size=32,
    num_workers=8,
    drop_last=False,
    shuffle=False,
    pin_memory=True,
    persistent_workers=True,
)

In [24]:
class Model(nn.Module):
    def __init__(self, num_classes):
        super(Model, self).__init__()
        self.net = ConvNextV2ForImageClassification.from_pretrained(
            "facebook/convnextv2-base-22k-224", drop_path_rate=0.5
        )
        self.net.classifier = nn.Identity()
        self.fc = nn.Linear(1024, 512, bias=False)
        self.head = nn.Linear(512, num_classes, bias=False)
        self.s = math.sqrt(2) * math.log(num_classes - 1)

    def forward(self, x):
        embeds = self.fc(self.net(pixel_values=x).logits)
        embeds = F.normalize(embeds, p=2, dim=-1)
        weight = F.normalize(self.head.weight, p=2, dim=-1)
        logits = F.linear(embeds * self.s, weight)
        logits = logits.float()
        return embeds, logits

In [25]:
model = Model(len(class2id))
model.to(device)

Loading weights:   0%|          | 0/380 [00:00<?, ?it/s]

Model(
  (net): ConvNextV2ForImageClassification(
    (convnextv2): ConvNextV2Model(
      (embeddings): ConvNextV2Embeddings(
        (patch_embeddings): Conv2d(3, 128, kernel_size=(4, 4), stride=(4, 4))
        (layernorm): ConvNextV2LayerNorm((128,), eps=1e-06, elementwise_affine=True)
      )
      (encoder): ConvNextV2Encoder(
        (stages): ModuleList(
          (0): ConvNextV2Stage(
            (downsampling_layer): ModuleList()
            (layers): ModuleList(
              (0): ConvNextV2Layer(
                (dwconv): Conv2d(128, 128, kernel_size=(7, 7), stride=(1, 1), padding=(3, 3), groups=128)
                (layernorm): ConvNextV2LayerNorm((128,), eps=1e-06, elementwise_affine=True)
                (pwconv1): Linear(in_features=128, out_features=512, bias=True)
                (act): GELUActivation()
                (grn): ConvNextV2GRN()
                (pwconv2): Linear(in_features=512, out_features=128, bias=True)
                (drop_path): Identity()
         

In [26]:
summary(model, input_size=(1, 3, 224, 224))  # 11,966,019

Layer (type:depth-idx)                                                 Output Shape              Param #
Model                                                                  [1, 512]                  7,482,880
├─ConvNextV2ForImageClassification: 1-1                                [1, 1024]                 --
│    └─ConvNextV2Model: 2-1                                            [1, 1024]                 --
│    │    └─ConvNextV2Embeddings: 3-1                                  [1, 128, 56, 56]          6,528
│    │    └─ConvNextV2Encoder: 3-2                                     [1, 1024, 7, 7]           87,684,224
│    │    └─LayerNorm: 3-3                                             [1, 1024]                 2,048
│    └─Identity: 2-2                                                   [1, 1024]                 --
├─Linear: 1-2                                                          [1, 512]                  524,288
Total params: 95,699,968
Trainable params: 95,699,968
Non-trainable p

In [27]:
model.compile(mode="reduce-overhead", dynamic=False, fullgraph=True)

In [28]:
muon_params = [
    p
    for name, p in model.named_parameters()
    if name.startswith("net.")
    and (
        name.rstrip("weight").endswith(
            ("pwconv1.", "pwconv2.", "downsampling_layer.1.")
        )
        and not print(name)
    )
]

adam_params = [
    p
    for name, p in model.named_parameters()
    if name.startswith("net.")
    and not (
        name.rstrip("weight").endswith(
            ("pwconv1.", "pwconv2.", "downsampling_layer.1.")
        )
    )
]

optimizer = Muon(
    [
        {"params": muon_params, "lr": 1e-4, "use_muon": True},
        {"params": adam_params, "lr": 1e-4, "use_muon": False},
        {"params": model.fc.weight, "lr": 1e-3, "use_muon": True},
        {"params": model.head.weight, "lr": 1e-3, "use_muon": False},
    ]
)

net.convnextv2.encoder.stages.0.layers.0.pwconv1.weight
net.convnextv2.encoder.stages.0.layers.0.pwconv2.weight
net.convnextv2.encoder.stages.0.layers.1.pwconv1.weight
net.convnextv2.encoder.stages.0.layers.1.pwconv2.weight
net.convnextv2.encoder.stages.0.layers.2.pwconv1.weight
net.convnextv2.encoder.stages.0.layers.2.pwconv2.weight
net.convnextv2.encoder.stages.1.downsampling_layer.1.weight
net.convnextv2.encoder.stages.1.layers.0.pwconv1.weight
net.convnextv2.encoder.stages.1.layers.0.pwconv2.weight
net.convnextv2.encoder.stages.1.layers.1.pwconv1.weight
net.convnextv2.encoder.stages.1.layers.1.pwconv2.weight
net.convnextv2.encoder.stages.1.layers.2.pwconv1.weight
net.convnextv2.encoder.stages.1.layers.2.pwconv2.weight
net.convnextv2.encoder.stages.2.downsampling_layer.1.weight
net.convnextv2.encoder.stages.2.layers.0.pwconv1.weight
net.convnextv2.encoder.stages.2.layers.0.pwconv2.weight
net.convnextv2.encoder.stages.2.layers.1.pwconv1.weight
net.convnextv2.encoder.stages.2.layers.1

In [29]:
WARMUP_EPOCHS = 1
EPOCHS = 7

In [30]:
main_scheduler = torch.optim.lr_scheduler.LinearLR(
    optimizer,
    start_factor=1.0,
    end_factor=0.0,
    total_iters=EPOCHS * len(train_dataloader),
)
warmup_scheduler = torch.optim.lr_scheduler.LinearLR(
    optimizer,
    start_factor=0.01,
    end_factor=1.0,
    total_iters=WARMUP_EPOCHS * len(train_dataloader),
)
scheduler = torch.optim.lr_scheduler.SequentialLR(
    optimizer,
    schedulers=[warmup_scheduler, main_scheduler],
    milestones=[WARMUP_EPOCHS * len(train_dataloader)],
)

In [ ]:
for epoch in tqdm(range(WARMUP_EPOCHS + EPOCHS)):
    print(f"EPOCH {epoch}:")

    model.train()

    running_loss = 0.0

    tbar = tqdm(total=len(train_dataloader))
    for x, labels, weights in train_dataloader:
        x = x.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        weights = weights.to(device, non_blocking=True)

        with torch.amp.autocast("cuda", dtype=torch.bfloat16):
            _, logits = model(x)
        loss = (F.cross_entropy(logits, labels, reduction="none") * weights).mean()

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        scheduler.step()

        running_loss += loss.detach()
        tbar.update(1)

    tbar.close()

    running_loss /= len(train_dataloader)
    print(f"Train Loss: {running_loss.item()}")

    model.eval()

    y_pred = []

    with torch.inference_mode():
        with torch.amp.autocast("cuda", dtype=torch.bfloat16):
            for x in val_dataloader:
                x = x.to(device, non_blocking=True)
                embeds, _ = model(x)
                y_pred.append(embeds.clone())

    y_pred = torch.cat(y_pred).cpu()

    query_embeds = y_pred[: len(query_df)]
    gallery_embeds = y_pred[-len(gallery_df) :]

    s = (
        F.normalize(query_embeds, p=2, dim=-1)
        @ F.normalize(gallery_embeds, p=2, dim=-1).T
    ).numpy()
    print("Val mAP:", evaluate(query_df, gallery_df, s))

  0%|          | 0/8 [00:00<?, ?it/s]

EPOCH 0:


  0%|          | 0/3 [00:00<?, ?it/s]

Train Loss: 9.94310188293457
Val mAP: 0.08495975492524097
EPOCH 1:


  0%|          | 0/3 [00:00<?, ?it/s]

Train Loss: 9.012626647949219
Val mAP: 0.07803665685689314
EPOCH 2:


  0%|          | 0/3 [00:00<?, ?it/s]

Train Loss: 7.966396331787109


In [ ]:
model.net.save_pretrained("./tmp_8")